# A1 · L1 Seed Vocabulary Extraction

**Track A · Week 1 D1–D3 · Colab CPU(免费 runtime 即可)**

Pipeline: 9 规则文件 → DeepSeek-V3 JSON-mode 抽 V/S/M/C → sentence-transformers 语义去重 → 频次×多源 排序 → `ontology_v1_seed.json`。

Run-all 即可。中途断线:`extract_primitives.jsonl` 支持断点续跑。

In [ ]:
# ========== 0. Colab setup ==========
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, json, re, time, hashlib, collections
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

PHYRE_ROOT  = Path('/content/drive/MyDrive/phyre')
RULES_DIR   = PHYRE_ROOT / 'data' / 'echem_rules' / 'staging'
OUT_VOCAB   = PHYRE_ROOT / 'data' / 'ontology_v1_seed.json'
LOG_EXTRACT = PHYRE_ROOT / 'logs' / 'A1_extract.jsonl'
LOG_REVIEW  = PHYRE_ROOT / 'logs' / 'A1_review.md'
for p in [OUT_VOCAB.parent, LOG_EXTRACT.parent]:
    p.mkdir(parents=True, exist_ok=True)

os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
!pip -q install openai sentence-transformers

In [ ]:
# ========== 1. load rule files — 多路径查找 + 友好诊断 ==========
from pathlib import Path
import shutil

# 候选路径(按优先级)
CANDIDATES = [
    PHYRE_ROOT / 'data' / 'echem_rules' / 'staging',          # 标准位置
    Path('/content/pvgap_experiment/data/echem_rules/staging'),  # 用户手动上传 zip 解压
    Path('/content/phyre-paper1/pvgap_experiment/data/echem_rules/staging'),  # git clone
]

RULES_DIR = None
for cand in CANDIDATES:
    if cand.is_dir() and list(cand.glob('*.jsonl')):
        RULES_DIR = cand; break

if RULES_DIR is None:
    print('✗ 规则文件未找到。有三个方式补齐(任选一):')
    print()
    print('方式 A — 本地上传到 Drive(最简单,一次到位):')
    print('  本地打开 Google Drive 网页,把整个目录')
    print('    pvgap_experiment/data/echem_rules/staging/')
    print('  上传到')
    print('    /MyDrive/phyre/data/echem_rules/staging/')
    print()
    print('方式 B — git clone(需先把项目 push 到 GitHub):')
    print('  !git clone https://github.com/<USER>/phyre-paper1.git /content/phyre-paper1')
    print()
    print('方式 C — Colab Files 面板直接拖 8 个 .jsonl 文件:')
    ups_dir = PHYRE_ROOT / 'data' / 'echem_rules' / 'staging'
    ups_dir.mkdir(parents=True, exist_ok=True)
    print(f'  目标位置:{ups_dir}')
    print(f'  然后重跑此 cell')
    raise SystemExit('rule files missing — 按上面指引补齐后重跑此 cell')

files = sorted(RULES_DIR.glob('*.jsonl'))
print(f'✓ found {len(files)} rule files at  {RULES_DIR}')
rules = []
for f in files:
    print(f'  {f.name}')
    for line in open(f, encoding='utf-8'):
        try:
            r = json.loads(line)
            r['_source'] = f.stem
            r['_rid'] = hashlib.md5((f.stem + line).encode()).hexdigest()[:10]
            rules.append(r)
        except Exception:
            pass
print(f'
total rules: {len(rules)}')
if rules:
    print(f'example: {json.dumps(rules[0], ensure_ascii=False, indent=2)[:400]}')
else:
    print('⚠ files found but 0 parseable rules — 检查 jsonl 格式')

In [ ]:
# ========== 1b. augment with §9E.1 benchmark synthesized rules (L1+L2+L3+L4) ==========
import hashlib, json, urllib.request
from pathlib import Path

BENCH_LOCAL_CANDIDATES = [
    Path('/content/phyre-paper1/pvgap_experiment/data/benchmark/echem_reason_benchmark.jsonl'),
    Path('/content/pvgap_experiment/data/benchmark/echem_reason_benchmark.jsonl'),
    PHYRE_ROOT / 'pvgap_experiment' / 'data' / 'benchmark' / 'echem_reason_benchmark.jsonl',
]
BENCH_URL = 'https://raw.githubusercontent.com/wjtwzgl12/main/pvgap_experiment/data/benchmark/echem_reason_benchmark.jsonl'.replace('/main/', '-LLM/main/')  # → wjtwzgl12/reasoning-LLM
BENCH_URL = 'https://raw.githubusercontent.com/wjtwzgl12/reasoning-LLM/main/pvgap_experiment/data/benchmark/echem_reason_benchmark.jsonl'

bench_text = None
for cand in BENCH_LOCAL_CANDIDATES:
    if cand.is_file():
        try:
            bench_text = cand.read_text(encoding='utf-8')
            print(f'[augment] loaded benchmark from local: {cand}')
            break
        except Exception as e:
            print(f'[augment] failed reading {cand}: {e}')
if bench_text is None:
    try:
        with urllib.request.urlopen(BENCH_URL, timeout=30) as resp:
            bench_text = resp.read().decode('utf-8')
        print(f'[augment] fetched benchmark from URL: {BENCH_URL}')
    except Exception as e:
        print(f'[augment] WARNING: could not fetch benchmark ({e}); skipping augmentation')
        bench_text = None

def _compose_rule_text(entry):
    """Compose a rule sentence from §9E.1 entry (L1 has answer; L2/3/4 have structured GT)."""
    gt = entry.get('ground_truth') or {}
    ki = entry.get('knowledge_ingredients') or []
    ans = gt.get('answer')
    if isinstance(ans, str) and ans.strip():
        return ans.strip()
    diag = (gt.get('diagnosis') or '').strip()
    dtyp = (gt.get('degradation_type') or '').strip()
    sev  = (gt.get('severity') or '').strip()
    parts = []
    if diag: parts.append(f'Diagnosis: {diag}')
    if dtyp: parts.append(f'mechanism={dtyp}')
    if sev:  parts.append(f'severity={sev}')
    if ki:   parts.append('knowledge_ingredients=[' + ', '.join(ki) + ']')
    eis = gt.get('eis_features') or {}
    if not eis:
        s2 = (gt.get('stages') or {}).get('S2_feature_extraction') or {}
        eis = s2 if isinstance(s2, dict) else {}
    sig_keys = ['R_ohm','peak_imag_freq_Hz','peak_freq','peak_neg_imag',
                'lf_slope','total_impedance_range','total_range']
    sig = {k: eis.get(k) for k in sig_keys if isinstance(eis.get(k), (int, float))}
    if sig:
        parts.append('EIS signature: ' + ', '.join(f'{k}={v:.4g}' for k,v in sig.items()))
    noop = gt.get('noop_text')
    if isinstance(noop, str) and noop.strip():
        parts.append(f'irrelevant_distractor="{noop.strip()}"')
    return '; '.join(parts) if parts else ''

if bench_text:
    try: rules
    except NameError: rules = []
    existing_rids = {r.get('_rid') for r in rules}
    n_synth = 0
    n_skipped_lvl = 0
    n_skipped_empty = 0
    by_lvl = {1:0, 2:0, 3:0, 4:0}
    for idx, line in enumerate(bench_text.splitlines()):
        line = line.strip()
        if not line: continue
        try: entry = json.loads(line)
        except Exception: continue
        level = entry.get('level')
        if level not in (1, 2, 3, 4):  # ← include L1 (was excluded before)
            n_skipped_lvl += 1; continue
        rule_text = _compose_rule_text(entry)
        if not rule_text:
            n_skipped_empty += 1; continue
        rid = hashlib.md5(rule_text.encode('utf-8')).hexdigest()[:12]
        if rid in existing_rids: continue
        qid = entry.get('qid') or f'q{idx}'
        rec = {
            'rule_id': f'synth_L{level}_{qid}',
            'rule': rule_text,
            'level': int(level),
            'sources': ['9E.1_synth'],
            '_rid': rid,
            '_source': '9E.1_benchmark_synth',
        }
        rules.append(rec); existing_rids.add(rid); n_synth += 1
        by_lvl[level] += 1
    print(f'[augment] +{n_synth} synthesized rules from §9E.1 (L1={by_lvl[1]} L2={by_lvl[2]} L3={by_lvl[3]} L4={by_lvl[4]}),'
          f' total now {len(rules)}  (skipped: lvl={n_skipped_lvl}, empty={n_skipped_empty})')


In [ ]:
# ========== 2. LLM extraction — DeepSeek-V3 JSON mode ==========
from openai import OpenAI
client = OpenAI(api_key=os.environ['DEEPSEEK_API_KEY'],
                base_url='https://api.deepseek.com')

SYSTEM = """You are an electrochemistry ontology extractor. Given a mechanistic rule or
observation, decompose it into structured primitives under this grammar:

  node = ⟨Verb, Substrate, Modifier*, Condition*⟩

Categories:
  V (Verb)      : process / action / transformation  (e.g. "dissolution", "growth",
                  "intercalation", "plating", "oxidation")
  S (Substrate) : material / species / interface     (e.g. "SEI", "NMC-cathode",
                  "Li-metal", "electrolyte", "LiF")
  M (Modifier)  : qualifier / kinetic regime         (e.g. "dendritic", "diffusion-
                  limited", "reversible", "catalysed")
  C (Condition) : thermodynamic / operational context (e.g. "high-SoC", "T>45C",
                  "high-rate", "low-voltage")

Rules:
  - Use lower-case, hyphenated English (e.g. "sei-growth", not "SEI Growth").
  - Each list may be empty. Do NOT invent; only extract what is mentioned or strongly
    implied.
  - Output STRICT JSON. Do not include commentary.
"""

FEW_SHOT = [
    {'role': 'user', 'content': 'Rule: "SEI grows thicker at high temperature, mostly via reduction of carbonate solvents at the anode surface."'},
    {'role': 'assistant', 'content': json.dumps({
        'V': ['growth', 'reduction'],
        'S': ['sei', 'carbonate-solvent', 'anode-surface'],
        'M': ['thickening'],
        'C': ['high-temperature']
    }, ensure_ascii=False)},
    {'role': 'user', 'content': 'Rule: "Li plating occurs on graphite at high C-rate when anode potential drops below 0 V vs Li/Li+."'},
    {'role': 'assistant', 'content': json.dumps({
        'V': ['plating'],
        'S': ['li-metal', 'graphite-anode'],
        'M': ['dendritic'],
        'C': ['high-c-rate', 'anode-potential<0V']
    }, ensure_ascii=False)},
    {'role': 'user', 'content': 'Rule: "Mn dissolution from spinel cathode accelerates above 3.0 V cutoff, poisoning the SEI on graphite."'},
    {'role': 'assistant', 'content': json.dumps({
        'V': ['dissolution', 'poisoning'],
        'S': ['mn-cation', 'spinel-cathode', 'sei', 'graphite-anode'],
        'M': ['transition-metal-crossover'],
        'C': ['high-voltage']
    }, ensure_ascii=False)},
]

def _rule_text(r):
    for k in ('rule', 'text', 'content', 'statement', 'body'):
        if isinstance(r.get(k), str) and r[k].strip(): return r[k]
    return json.dumps({k: v for k, v in r.items() if not k.startswith('_')},
                      ensure_ascii=False)[:800]

def extract_primitives(rule):
    text = _rule_text(rule)
    msgs = [{'role': 'system', 'content': SYSTEM}] + FEW_SHOT + [
        {'role': 'user', 'content': f'Rule: "{text}"'}]
    for attempt in range(3):
        try:
            r = client.chat.completions.create(
                model='deepseek-chat', messages=msgs,
                response_format={'type': 'json_object'},
                temperature=0.2, max_tokens=400)
            out = json.loads(r.choices[0].message.content)
            return {k: [s.strip().lower() for s in (out.get(k) or [])
                        if isinstance(s, str) and s.strip()]
                    for k in ('V','S','M','C')}
        except Exception as e:
            if attempt == 2: return {'V':[],'S':[],'M':[],'C':[], '_err': str(e)[:200]}
            time.sleep(2 ** attempt)

In [ ]:
# ========== 2b. LLM rule-expansion — per rule, generate 3 sub-mechanism variants ==========
# Take each rule and ask DeepSeek for 3 distinct *sub-mechanism* statements that
# decompose the parent mechanism into its constituent steps. Each sub-rule then
# goes through the same V/S/M/C extractor as the original — multiplying the
# extraction corpus 3-4× without sacrificing semantic quality. This is the real
# fix for the R<N underdetermined NOTEARS problem (was 65 → ~250+).

LOG_EXPAND = PHYRE_ROOT / 'logs' / 'A1_expand.jsonl'
LOG_EXPAND.parent.mkdir(parents=True, exist_ok=True)

EXPAND_SYSTEM = """You are an electrochemistry mechanism expander. Given a parent
electrochemistry rule or diagnosis, write 3 DISTINCT sub-mechanism statements that
decompose it into intermediate physical/chemical steps. Each sub-rule must:
  - be a single sentence (≤30 words);
  - mention concrete primitives (verb, substrate, modifier, condition);
  - cover a DIFFERENT step of the chain (not paraphrases of each other);
  - stay scientifically faithful — do not invent mechanisms not implied.

Output STRICT JSON: {"sub_rules": ["...", "...", "..."]}
"""

EXPAND_FEW_SHOT = [
    {'role': 'user', 'content': 'Parent rule: "SEI grows thicker at high T via carbonate reduction at the anode."'},
    {'role': 'assistant', 'content': json.dumps({'sub_rules': [
        'At elevated temperature electrolyte solvent reduction kinetics accelerate at the anode interface.',
        'Reduced carbonate species deposit on the SEI surface, increasing its thickness.',
        'A thicker SEI raises charge-transfer impedance and lowers Coulombic efficiency.'
    ]}, ensure_ascii=False)},
]

def expand_rule(rule):
    text = _rule_text(rule)[:600]
    msgs = [{'role': 'system', 'content': EXPAND_SYSTEM}] + EXPAND_FEW_SHOT + [
        {'role': 'user', 'content': f'Parent rule: "{text}"'}]
    for attempt in range(3):
        try:
            r = client.chat.completions.create(
                model='deepseek-chat', messages=msgs,
                response_format={'type': 'json_object'},
                temperature=0.5, max_tokens=400)
            obj = json.loads(r.choices[0].message.content)
            subs = obj.get('sub_rules') or []
            return [s.strip() for s in subs if isinstance(s, str) and s.strip()][:3]
        except Exception as e:
            if attempt == 2: return []
            time.sleep(2 ** attempt)

# resume support: skip rules whose parent_rid is already expanded
expanded_done = set()
if LOG_EXPAND.exists():
    for line in open(LOG_EXPAND, encoding='utf-8'):
        try: expanded_done.add(json.loads(line)['parent_rid'])
        except: pass
print(f'expansion resume: {len(expanded_done)} / {len(rules)} parents already expanded')

todo = [r for r in rules if r['_rid'] not in expanded_done]
n_new_subs = 0
with open(LOG_EXPAND, 'a', encoding='utf-8') as fh:
    with ThreadPoolExecutor(max_workers=10) as pool:
        futs = {pool.submit(expand_rule, r): r for r in todo}
        for i, fut in enumerate(as_completed(futs)):
            parent = futs[fut]
            subs = fut.result()
            for j, sub in enumerate(subs):
                fh.write(json.dumps({
                    'parent_rid': parent['_rid'],
                    'parent_source': parent['_source'],
                    'sub_idx': j,
                    'rule': sub,
                }, ensure_ascii=False) + '\n')
                n_new_subs += 1
            fh.flush()
            if (i+1) % 20 == 0:
                print(f'  expanded {i+1}/{len(todo)}  (subs so far: {n_new_subs})')

# Inject sub-rules back into the `rules` list so cell-5 extracts primitives from them too
existing_rids = {r['_rid'] for r in rules}
n_added = 0
for line in open(LOG_EXPAND, encoding='utf-8'):
    try: rec = json.loads(line)
    except: continue
    sub_text = rec['rule']
    rid = hashlib.md5(sub_text.encode('utf-8')).hexdigest()[:12]
    if rid in existing_rids: continue
    rules.append({
        'rule_id': f'sub_{rec["parent_rid"]}_{rec["sub_idx"]}',
        'rule': sub_text,
        'level': 0,                         # synthetic level
        'sources': ['llm_expand'],
        '_rid': rid,
        '_source': f'expand_of_{rec["parent_source"]}',
    })
    existing_rids.add(rid); n_added += 1
print(f'\n[expand] +{n_added} sub-rules injected, total rules now {len(rules)}')


In [ ]:
# ========== 3. concurrent extraction with resume ==========
done = set()
if LOG_EXTRACT.exists():
    for line in open(LOG_EXTRACT, encoding='utf-8'):
        try: done.add(json.loads(line)['_rid'])
        except: pass
print(f'resume: {len(done)} / {len(rules)} already done')

todo = [r for r in rules if r['_rid'] not in done]
with open(LOG_EXTRACT, 'a', encoding='utf-8') as fh:
    with ThreadPoolExecutor(max_workers=10) as pool:
        futs = {pool.submit(extract_primitives, r): r for r in todo}
        for i, fut in enumerate(as_completed(futs)):
            r = futs[fut]
            prim = fut.result()
            rec = {'_rid': r['_rid'], '_source': r['_source'], **prim}
            fh.write(json.dumps(rec, ensure_ascii=False) + '\n'); fh.flush()
            if (i+1) % 20 == 0: print(f'  extracted {i+1}/{len(todo)}')
print('extraction done')

In [ ]:
# ========== 4. synonym merge via sentence-transformers + union-find ==========
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

extracted = [json.loads(l) for l in open(LOG_EXTRACT, encoding='utf-8')]
print(f'loaded {len(extracted)} extraction records')

# aggregate per category: term → {freq, sources}
per_cat = {'V': {}, 'S': {}, 'M': {}, 'C': {}}
for rec in extracted:
    for k in 'VSMC':
        for term in rec.get(k, []):
            d = per_cat[k].setdefault(term, {'freq': 0, 'sources': set()})
            d['freq'] += 1
            d['sources'].add(rec['_source'])

for k in 'VSMC': print(f'  {k}: {len(per_cat[k])} raw terms')

def union_find_merge(terms, threshold=0.88):
    """Cosine-cluster terms; return {representative: [member, ...]}."""
    if not terms: return {}
    embs = model.encode(terms, normalize_embeddings=True, show_progress_bar=False)
    sim = embs @ embs.T
    n = len(terms)
    parent = list(range(n))
    def find(i):
        while parent[i] != i: parent[i] = parent[parent[i]]; i = parent[i]
        return i
    for i in range(n):
        for j in range(i+1, n):
            if sim[i, j] >= threshold:
                ri, rj = find(i), find(j)
                if ri != rj: parent[rj] = ri
    clusters = collections.defaultdict(list)
    for i in range(n): clusters[find(i)].append(terms[i])
    # Pick representative: shortest term in cluster (usually canonical form)
    return {min(v, key=len): v for v in clusters.values()}

merged = {}
for k in 'VSMC':
    terms = list(per_cat[k].keys())
    clusters = union_find_merge(terms, threshold=0.88)
    merged[k] = {}
    for rep, members in clusters.items():
        freq = sum(per_cat[k][m]['freq'] for m in members)
        sources = set().union(*[per_cat[k][m]['sources'] for m in members])
        merged[k][rep] = {'freq': freq, 'sources': sorted(sources),
                          'aliases': sorted(m for m in members if m != rep)}
    print(f'  {k}: {len(terms)} → {len(merged[k])} merged')

In [ ]:
# ========== 5. rank and pick top-K ==========
TARGETS = {'V': 15, 'S': 20, 'M': 12, 'C': 10}

def score(d): return np.log1p(d['freq']) * (1 + len(d['sources']))

vocab_v1 = {'version': 'v1_seed',
            'frozen_grammar': {
                'node_schema': ['Verb', 'Substrate', 'Modifier*', 'Condition*'],
                'relations': ['triggers', 'rate-limits', 'competes-with',
                              'co-occurs', 'amplifies', 'suppresses']
            },
            'vocab': {}}

for k, target in TARGETS.items():
    ranked = sorted(merged[k].items(), key=lambda kv: -score(kv[1]))
    # take up to max(target, ranked with score>=median) to avoid over-pruning
    take = max(target, sum(1 for _, d in ranked if d['freq'] >= 2))
    vocab_v1['vocab'][k] = [
        {'name': name, 'freq': d['freq'], 'sources': d['sources'],
         'aliases': d['aliases'], 'def': ''}   # def 留空,TASK 6 人工补
        for name, d in ranked[:take]
    ]
    print(f'  {k}: kept {take} (target {target})')

In [ ]:
# ========== 6. LLM-drafted definitions for each entry ==========
# 为每个词条生成 1 句定义,供你审阅
DEF_SYSTEM = """You are an electrochemistry glossary writer. Given a primitive
term and its category (V/S/M/C), write ONE concise definition (≤20 words) that
states what it is and typical experimental signature. Output plain text only."""

def def_one(cat, name):
    try:
        r = client.chat.completions.create(
            model='deepseek-chat', temperature=0.0, max_tokens=80,
            messages=[{'role': 'system', 'content': DEF_SYSTEM},
                      {'role': 'user', 'content': f'Category: {cat}\nTerm: {name}'}])
        return r.choices[0].message.content.strip()
    except Exception as e: return f'(def-err: {e})'

with ThreadPoolExecutor(max_workers=8) as pool:
    for cat in 'VSMC':
        futs = {pool.submit(def_one, cat, e['name']): e for e in vocab_v1['vocab'][cat]}
        for fut in as_completed(futs):
            futs[fut]['def'] = fut.result()
        print(f'  {cat} definitions done')

In [ ]:
# ========== 7. write vocab + human-review markdown ==========
with open(OUT_VOCAB, 'w', encoding='utf-8') as f:
    json.dump(vocab_v1, f, indent=2, ensure_ascii=False)
print(f'wrote {OUT_VOCAB}')

lines = ['# A1 Seed Vocab — human review\n',
         f'Generated {time.strftime("%Y-%m-%d %H:%M")} · target |V|=15 |S|=20 |M|=12 |C|=10\n']
for cat in 'VSMC':
    lines.append(f'\n## {cat}  ({len(vocab_v1["vocab"][cat])} entries)\n')
    lines.append('| name | freq | sources | def | aliases |')
    lines.append('|---|---|---|---|---|')
    for e in vocab_v1['vocab'][cat]:
        aliases = ', '.join(e['aliases'][:3]) + ('...' if len(e['aliases']) > 3 else '')
        srcs = ', '.join(e['sources'][:2]) + ('...' if len(e['sources']) > 2 else '')
        lines.append(f'| `{e["name"]}` | {e["freq"]} | {srcs} | {e["def"]} | {aliases} |')
LOG_REVIEW.write_text('\n'.join(lines), encoding='utf-8')
print(f'wrote {LOG_REVIEW}  (open in Drive, spot-check 20 entries, then sign off)')

# 签字确认后:  git add data/ontology_v1_seed.json && git commit -m 'A1: seed vocab v1 frozen' && git tag w1-vocab-done

In [ ]:
# cell-9: diagnostic + auto-recover from incomplete extraction runs
ext_rids = set()
if LOG_EXTRACT.exists():
    for l in open(LOG_EXTRACT, encoding='utf-8'):
        try: ext_rids.add(json.loads(l)['_rid'])
        except: pass
missing = [r for r in rules if r['_rid'] not in ext_rids]
print(f'  rules total      = {len(rules)}')
print(f'  extracted before = {len(ext_rids)}')
print(f'  missing now      = {len(missing)}')

if len(missing) > 20:
    print(f'
⚠ extraction was incomplete — re-running on {len(missing)} missing rules (workers=4)')
    with open(LOG_EXTRACT, 'a', encoding='utf-8') as fh:
        with ThreadPoolExecutor(max_workers=4) as pool:
            futs = {pool.submit(extract_primitives, r): r for r in missing}
            for i, fut in enumerate(as_completed(futs)):
                r = futs[fut]; prim = fut.result()
                rec = {'_rid': r['_rid'], '_source': r['_source'], **prim}
                fh.write(json.dumps(rec, ensure_ascii=False) + '
'); fh.flush()
                if (i+1) % 10 == 0: print(f'    recovered {i+1}/{len(missing)}')
    # re-tally
    ext_rids2 = set()
    for l in open(LOG_EXTRACT, encoding='utf-8'):
        try: ext_rids2.add(json.loads(l)['_rid'])
        except: pass
    print(f'  after recovery: {len(ext_rids2)} / {len(rules)} extracted')
elif missing:
    print(f'
  only {len(missing)} missing — leaving as-is (recovery threshold > 20)')
else:
    print(f'
  ✓ extraction complete')

# Independently verify merged vocab size
if OUT_VOCAB.exists():
    v = json.loads(OUT_VOCAB.read_text(encoding='utf-8'))
    sizes = {k: len(v['vocab'][k]) for k in 'VSMC'}
    print(f'
  ontology_v1_seed sizes: {sizes}')
    target = {'V': 15, 'S': 20, 'M': 12, 'C': 10}
    if all(sizes[k] >= int(target[k] * 0.8) for k in 'VSMC'):
        print('  ✓ vocab sizes within 80% of target — A2/A3 can consume')
    else:
        short = [k for k in 'VSMC' if sizes[k] < int(target[k] * 0.8)]
        print(f'  ⚠ undersized cats {short} — re-run cells 4-7 to remerge with full extract')
else:
    print('  ⚠ ontology_v1_seed.json not found — re-run cells 4-7')


## Diagnostic + auto-recovery
This cell idempotently checks extraction coverage. If >20 rules missing (e.g. due to a quota burst or session timeout), it re-runs the LLM on just the missing rids with reduced concurrency. Run this cell after every fresh A1 run; safe to re-run.